# Workstream 2 — Metric Validation

This notebook independently reads the **processed output files** created by the
Housing Need, Demand & Affordability notebook and returns one validation value
for every metric in the Workstream 2 validation sheet.

Important period rule:
- The current ACS pipeline is **2024 ACS 5-year (2020–2024)**, so ACS rows are
  not assigned arbitrary 2018–2025 years.
- HUD rows use **FY 2026**.
- NHPD rows remain **data pending** while `INCLUDE_NHPD=False`.
- Population, median home value, and median sale price are identified explicitly
  when they are not produced by the current notebook rather than fabricating values.


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np


In [4]:
from pathlib import Path

def find_workstream_root():
    """Find the Workstream 2 folder containing data/ and notebooks."""
    
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:

        # Case 1: notebook is already inside the Workstream 2 folder
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

        # Case 2: repo contains a folder named "lauren's work"
        lauren_folder = candidate / "lauren's work"

        if (
            (lauren_folder / "data").exists()
            and (lauren_folder / "notebooks").exists()
        ):
            return lauren_folder

    raise FileNotFoundError(
        "Could not locate the Workstream 2 folder containing "
        "data/ and notebooks/."
    )


ROOT = find_workstream_root()
PROCESSED_DIR = ROOT / "data" / "processed"

print("Workstream root:")
print(ROOT)

print("\nProcessed folder:")
print(PROCESSED_DIR)

Workstream root:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work

Processed folder:
/Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed


In [5]:
# Load the exact processed outputs created by the original notebook.

combined_path = (
    PROCESSED_DIR / "housing_need_affordability_acs2024_hud2026.csv"
)
nhpd_summary_path = (
    PROCESSED_DIR / "nhpd_san_diego_by_jurisdiction.csv"
)
nhpd_subsidy_path = (
    PROCESSED_DIR / "nhpd_san_diego_subsidies.csv"
)

if not combined_path.exists():
    raise FileNotFoundError(
        f"Missing {combined_path.name}. Run the original Workstream 2 notebook first."
    )

combined = pd.read_csv(combined_path)

nhpd_summary = (
    pd.read_csv(nhpd_summary_path)
    if nhpd_summary_path.exists()
    else pd.DataFrame()
)

nhpd_subsidies = (
    pd.read_csv(nhpd_subsidy_path)
    if nhpd_subsidy_path.exists()
    else pd.DataFrame()
)

print("Combined ACS/HUD rows:", len(combined))
print("Combined columns:", len(combined.columns))
print("NHPD summary rows:", len(nhpd_summary))
print("NHPD subsidy rows:", len(nhpd_subsidies))


Combined ACS/HUD rows: 19
Combined columns: 78
NHPD summary rows: 0
NHPD subsidy rows: 0


In [6]:
# Exact validation plan, in the same order as the metric spreadsheet.

validation_plan = pd.DataFrame([
    ["Population", "Carlsbad", "2020–2024 ACS 5-year", "unsupported", None],
    ["Median Household Income", "Encinitas", "2020–2024 ACS 5-year", "acs", "median_household_income"],
    ["San Diego County Median Household Income", "San Diego County (Countywide)", "2020–2024 ACS 5-year", "acs", "county_median_household_income"],
    ["Jurisdiction / County Income Ratio", "National City", "2020–2024 ACS 5-year", "acs", "jurisdiction_county_income_ratio"],
    ["Renter Cost-Burdened Households - Count", "El Cajon", "2020–2024 ACS 5-year", "acs", "renter_cost_burdened_count"],
    ["Renter Cost-Burdened Households - Percent", "Lemon Grove", "2020–2024 ACS 5-year", "acs", "renter_cost_burdened_pct"],
    ["Severely Cost-Burdened Renters - Count", "Oceanside", "2020–2024 ACS 5-year", "acs", "renter_severely_burdened_count"],
    ["Severely Cost-Burdened Renters - Percent", "San Marcos", "2020–2024 ACS 5-year", "acs", "renter_severely_burdened_pct"],
    ["Homeowner Cost-Burdened Households - Count", "Poway", "2020–2024 ACS 5-year", "acs", "owner_cost_burdened_count"],
    ["Homeowner Cost-Burdened Households - Percent", "Chula Vista", "2020–2024 ACS 5-year", "acs", "owner_cost_burdened_pct"],
    ["Severely Cost-Burdened Homeowners - Count", "Coronado", "2020–2024 ACS 5-year", "acs", "owner_severely_burdened_count"],
    ["Severely Cost-Burdened Homeowners - Percent", "Solana Beach", "2020–2024 ACS 5-year", "acs", "owner_severely_burdened_pct"],
    ["Homeowner Cost Burden - With Mortgage", "San Diego", "2020–2024 ACS 5-year", "acs", "owner_with_mortgage_burdened_pct"],
    ["Homeowner Cost Burden - Without Mortgage", "La Mesa", "2020–2024 ACS 5-year", "acs", "owner_without_mortgage_burdened_pct"],
    ["Median Gross Rent - 1 Bedroom", "Del Mar", "2020–2024 ACS 5-year", "acs", "median_gross_rent_1br"],
    ["1BR Rent-to-Income Percent", "Imperial Beach", "2020–2024 ACS 5-year", "acs", "one_br_rent_to_income_pct"],
    ["Median Owner Costs - With Mortgage", "Lemon Grove", "2020–2024 ACS 5-year", "acs", "median_owner_cost_with_mortgage"],
    ["Median Owner Costs - Without Mortgage", "Encinitas", "2020–2024 ACS 5-year", "acs", "median_owner_cost_without_mortgage"],
    ["Owner-Cost-to-Income Percent", "Carlsbad", "2020–2024 ACS 5-year", "acs", "owner_cost_with_mortgage_to_income_pct"],
    ["Median Home Value", "Vista", "2020–2024 ACS 5-year", "unsupported", None],
    ["Median Sale Price", "Santee", "Not in current notebook", "unsupported", None],
    ["Federally Assisted Properties", "San Diego", "NHPD data pending", "nhpd_summary", "federally_assisted_properties"],
    ["Federally Assisted Units", "Chula Vista", "NHPD data pending", "nhpd_summary", "total_units_in_assisted_properties"],
    ["Federal Housing Program", "Oceanside", "NHPD data pending", "nhpd_subsidy", "program_name"],
    ["Affordability / Subsidy Status", "Escondido", "NHPD data pending", "nhpd_subsidy", "subsidy_status"],
    ["Affordability Expiration Date", "National City", "NHPD data pending", "nhpd_subsidy", "subsidy_end_date"],
    ["At-Risk Assisted Properties", "El Cajon", "NHPD 5-year risk window — data pending", "nhpd_summary", "properties_expiring_within_5_years"],
    ["At-Risk Assisted Units", "Vista", "NHPD 5-year risk window — data pending", "nhpd_summary", "units_in_expiring_properties"],
    ["4-Person HUD Median Family Income", "San Diego County (Regional HUD benchmark)", "HUD FY 2026 (effective 2026-05-01)", "hud", "hud_four_person_median_family_income"],
    ["4-Person Extremely Low Income Limit", "San Diego County (Regional HUD benchmark)", "HUD FY 2026 (effective 2026-05-01)", "hud", "hud_four_person_extremely_low_income_limit"],
    ["4-Person Very Low Income Limit", "San Diego County (Regional HUD benchmark)", "HUD FY 2026 (effective 2026-05-01)", "hud", "hud_four_person_very_low_income_limit"],
    ["4-Person Low Income Limit", "San Diego County (Regional HUD benchmark)", "HUD FY 2026 (effective 2026-05-01)", "hud", "hud_four_person_low_income_limit"],
], columns=["Metric", "Jurisdiction", "Year / Period", "_source_type", "_column"])

display(validation_plan[["Metric", "Jurisdiction", "Year / Period"]])


,Metric,Jurisdiction,Year / Period
0,Population,Carlsbad,2020–2024 ACS 5-year
1,Median Household Income,Encinitas,2020–2024 ACS 5-year
2,San Diego County Median Household Income,San Diego County (Countywide),2020–2024 ACS 5-year
3,Jurisdiction / County Income Ratio,National City,2020–2024 ACS 5-year
4,Renter Cost-Burdened Households - Count,El Cajon,2020–2024 ACS 5-year
5,Renter Cost-Burdened Households - Percent,Lemon Grove,2020–2024 ACS 5-year
6,Severely Cost-Burdened Renters - Count,Oceanside,2020–2024 ACS 5-year
7,Severely Cost-Burdened Renters - Percent,San Marcos,2020–2024 ACS 5-year
8,Homeowner Cost-Burdened Households - Count,Poway,2020–2024 ACS 5-year
9,Homeowner Cost-Burdened Households - Percent,Chula Vista,2020–2024 ACS 5-year


In [7]:
def acs_value(jurisdiction, column):
    rows = combined[combined["jurisdiction"].eq(jurisdiction)]

    if len(rows) != 1:
        return f"ERROR: expected 1 row, found {len(rows)}"

    if column not in combined.columns:
        return f"MISSING COLUMN: {column}"

    return rows.iloc[0][column]


def hud_value(column):
    if column not in combined.columns:
        return f"MISSING COLUMN: {column}"

    values = combined[column].dropna().unique()

    if len(values) != 1:
        return f"ERROR: expected 1 HUD value, found {len(values)}"

    return values[0]


def nhpd_summary_value(jurisdiction, column):
    if nhpd_summary.empty:
        return "DATA PENDING — NHPD not loaded"

    if column not in nhpd_summary.columns:
        return f"MISSING COLUMN: {column}"

    rows = nhpd_summary[
        nhpd_summary["jurisdiction"].eq(jurisdiction)
    ]

    if rows.empty:
        return "NO MATCHING NHPD ROW"

    return rows.iloc[0][column]


def nhpd_subsidy_value(jurisdiction, column):
    if nhpd_subsidies.empty:
        return "DATA PENDING — NHPD not loaded"

    if column not in nhpd_subsidies.columns:
        return f"MISSING COLUMN: {column}"

    rows = nhpd_subsidies[
        nhpd_subsidies["jurisdiction"].eq(jurisdiction)
    ].copy()

    if rows.empty:
        return "NO MATCHING NHPD ROW"

    if column == "subsidy_end_date":
        rows[column] = pd.to_datetime(rows[column], errors="coerce")
        rows = rows.sort_values(column)

    values = rows[column].dropna()

    if values.empty:
        return "NO NON-MISSING VALUE"

    return values.iloc[0]


def validation_value(row):
    source_type = row["_source_type"]
    column = row["_column"]
    jurisdiction = row["Jurisdiction"]

    if source_type == "acs":
        return acs_value(jurisdiction, column)

    if source_type == "hud":
        return hud_value(column)

    if source_type == "nhpd_summary":
        return nhpd_summary_value(jurisdiction, column)

    if source_type == "nhpd_subsidy":
        return nhpd_subsidy_value(jurisdiction, column)

    if row["Metric"] == "Population":
        return "NOT IN CURRENT NOTEBOOK OUTPUT"

    if row["Metric"] == "Median Home Value":
        return "NOT IN CURRENT NOTEBOOK OUTPUT"

    if row["Metric"] == "Median Sale Price":
        return "NOT IN CURRENT NOTEBOOK OUTPUT"

    return "NOT AVAILABLE"


validation_output = validation_plan.copy()

validation_output["Notebook Value"] = validation_output.apply(
    validation_value,
    axis=1,
)

validation_output = validation_output[
    ["Metric", "Jurisdiction", "Year / Period", "Notebook Value"]
]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(validation_output)

assert len(validation_output) == 32
print("Validation rows:", len(validation_output))


,Metric,Jurisdiction,Year / Period,Notebook Value
0,Population,Carlsbad,2020–2024 ACS 5-year,NOT IN CURRENT NOTEBOOK OUTPUT
1,Median Household Income,Encinitas,2020–2024 ACS 5-year,162229.0
2,San Diego County Median Household Income,San Diego County (Countywide),2020–2024 ACS 5-year,106268.0
3,Jurisdiction / County Income Ratio,National City,2020–2024 ACS 5-year,0.629
4,Renter Cost-Burdened Households - Count,El Cajon,2020–2024 ACS 5-year,13157.0
5,Renter Cost-Burdened Households - Percent,Lemon Grove,2020–2024 ACS 5-year,61.82
6,Severely Cost-Burdened Renters - Count,Oceanside,2020–2024 ACS 5-year,7232.0
7,Severely Cost-Burdened Renters - Percent,San Marcos,2020–2024 ACS 5-year,34.48
8,Homeowner Cost-Burdened Households - Count,Poway,2020–2024 ACS 5-year,3264.0
9,Homeowner Cost-Burdened Households - Percent,Chula Vista,2020–2024 ACS 5-year,33.68


Validation rows: 32


In [8]:
# Save a copy-ready validation output.

output_path = (
    PROCESSED_DIR / "workstream2_metric_validation_output.csv"
)

validation_output.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: /Users/laurenvo/Documents/Github/chpd-dashboard-data-validation/lauren's work/data/processed/workstream2_metric_validation_output.csv


In [9]:
# Optional: print ONLY the Notebook Value column, one value per line,
# in the exact same row order as the validation spreadsheet.

for value in validation_output["Notebook Value"]:
    print(value)


NOT IN CURRENT NOTEBOOK OUTPUT
162229.0
106268.0
0.629
13157.0
61.82
7232.0
34.48
3264.0
33.68
860.0
29.37
37.7
12.86
3003.0
24.33
2538.0
993.0
31.45
NOT IN CURRENT NOTEBOOK OUTPUT
NOT IN CURRENT NOTEBOOK OUTPUT
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
DATA PENDING — NHPD not loaded
130900
52450
87450
139900
